# 시각화

In [ ]:
# 0) 필요한 패키지(최초 1회만)
# pip install geopandas shapely pyproj matplotlib contextily

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path
# 사용: p = Path("data/file.csv")
import math
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
# 1) 파일 경로 지정 (여기만 바꾸세요)

from IPython.display import display, HTML

def folium_deepnote_show(m):
    data = m.get_root().render()
    data_fixed_height = data.replace('width: 100%;height: 100%', 'width: 100%').replace('height: 100.0%;', 'height: 609px;', 1)
    display(HTML(data_fixed_height))


shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)

gdf_5179 = dunsan.to_crs(5179)

# 3) 육각형 하나를 만드는 헬퍼
def hexagon(cx, cy, r):
    # 정육각형 꼭짓점 (각도: 0, 60, 120, ...)
    coords = [(cx + r*math.cos(math.radians(a)),
               cy + r*math.sin(math.radians(a))) for a in range(0, 360, 60)]
    return Polygon(coords)

# 4) 경계 bbox 위로 육각 그리드 생성
def make_hex_grid(boundary_gdf, r=200):
    # r: 외접반지름(중심→꼭짓점 거리). 중심 간 가로 간격은 1.5*r, 세로 간격은 h = sqrt(3)*r/2 * 2 = sqrt(3)*r
    bounds = boundary_gdf.total_bounds  # (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = bounds
    width_step = 1.5 * r
    height_step = math.sqrt(3) * r

    # 약간 여유를 두고 시작/끝 좌표 결정
    x_start = minx - 2*r
    x_end   = maxx + 2*r
    y_start = miny - 2*r
    y_end   = maxy + 2*r

    hexes = []
    x = x_start
    col = 0
    while x <= x_end:
        # 홀짝 열마다 y 오프셋(반 칸) 줌
        y_offset = 0 if col % 2 == 0 else (height_step / 2.0)
        y = y_start + y_offset
        while y <= y_end:
            poly = hexagon(x, y, r)
            hexes.append(poly)
            y += height_step
        x += width_step
        col += 1

    grid = gpd.GeoDataFrame(geometry=hexes, crs=boundary_gdf.crs)
    # 5) 경계로 클리핑
    boundary_union = unary_union(boundary_gdf.geometry)
    clipped = gpd.clip(grid, gpd.GeoSeries([boundary_union], crs=boundary_gdf.crs))
    clipped = clipped.reset_index(drop=True)
    clipped["hex_id"] = clipped.index + 1
    return clipped

# 6) 격자 만들기 (R=200m 기본값; 필요시 변경)
R = 200  # m
hex_5179 = make_hex_grid(gdf_5179, r=R)

# 7) 중심점(centroid) 계산
centers_5179 = hex_5179.copy()
centers_5179["geometry"] = centers_5179.centroid
centers_5179 = centers_5179.set_geometry("geometry")

# 8) Folium/저장용으로 WGS84로 변환
hex_4326 = hex_5179.to_crs(4326)
centers_4326 = centers_5179.to_crs(4326)
centers_4326["lon"] = centers_4326.geometry.x
centers_4326["lat"] = centers_4326.geometry.y

# 9) (선택) 파일로 저장
# hex_4326.to_file("dunsan_hex_R200.geojson", driver="GeoJSON")
# centers_4326.to_file("dunsan_hex_centers_R200.geojson", driver="GeoJSON")

# 10) Folium 간단 시각화 예시
import folium
b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])

# 육각형(얇게)
folium.GeoJson(hex_4326.to_json(),
               name=f"Hex R={R}m",
               style_function=lambda f: {"color":"#555","weight":1,"fill":False}).add_to(m)

# 육각형 중심에 점 
for _, row in centers_4326.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2,
        color="#ff7f0e",
        fill=True,
        fill_color="#ff7f0e",
        fill_opacity=1.0,
    ).add_to(m)

m

## 수요지 시각화

In [ ]:
# ================== 필요한 패키지 ==================
# pip install geopandas shapely folium pyproj

import math
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
import folium

# ================== 1) 파라미터 ==================
CRS_M   = 5179      # 작업/거리계산용(미터)
CRS_LAT = 4326      # 지도 표시용
R       = 200       # 육각형 반경(m) (center→vertex)
MARGIN  = 0         # '온전한' 판정 여유(경계 내부로 MARGIN m 수축, 0이면 정확히 within)

# 선택옵션 -----------------------------
USE_FULL_WITHIN = False   # True: "온전한(완전 포함)" 육각형만 사용 / False: 센트로이드 기준 사용
CLIP_TO_BOUND   = True    # True: 경계로 육각형을 클립하여 라인 생성 / False: 원형 그대로 라인
CENTER_MARGIN   = 0       # 센트로이드 기준일 때, 경계에서 최소 이격(m). 0이면 경계 안이면 허용
# ---------------------------------------

# ================== 2) 경계 읽기 ==================
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')
gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 경계 데이터
dunsan = gdf[gdf['EMD_NM'] == '둔산동'].copy()
if dunsan.empty:
    raise ValueError("경계에서 '둔산동'이 발견되지 않았습니다. EMD_NM 값을 확인하세요.")

gdf_5179 = dunsan.to_crs(CRS_M)
gdf_4326 = dunsan.to_crs(CRS_LAT)

# ================== 3) 헬퍼 ==================
def hexagon(cx, cy, r):
    """중심(cx, cy)과 반경 r로 정육각형 폴리곤 생성(꼭짓점 각도 0, 60, 120, ...)."""
    return Polygon([
        (cx + r*math.cos(math.radians(a)), cy + r*math.sin(math.radians(a)))
        for a in range(0, 360, 60)
    ])

def make_hex_grid(boundary_gdf, r=200):
    """경계 bbox 위에 '빽빽한' 육각 그리드 생성(클립 전 원본 그리드 반환)"""
    minx, miny, maxx, maxy = boundary_gdf.total_bounds
    pad = r * 2
    minx, miny, maxx, maxy = minx - pad, miny - pad, maxx + pad, maxy + pad

    dx = 1.5 * r
    dy = math.sqrt(3) * r

    geoms = []
    col = 0
    x = minx
    while x <= maxx:
        y = miny + (0 if (col % 2 == 0) else dy / 2.0)
        while y <= maxy:
            geoms.append(hexagon(x, y, r))
            y += dy
        x += dx
        col += 1

    grid = gpd.GeoDataFrame(geometry=geoms, crs=boundary_gdf.crs)
    grid.reset_index(drop=True, inplace=True)
    grid["hex_id"] = grid.index + 1
    return grid

# ================== 4) 육각 그리드 생성 ==================
hex_grid_5179 = make_hex_grid(gdf_5179, r=R)

# 경계 단일화(멀티폴리곤이어도 overlay에서 그대로 사용 가능)
B = unary_union(gdf_5179.geometry)

# ================== 5) keep 집합(센트로이드 기준 vs 완전 포함) ==================
if USE_FULL_WITHIN:
    # '온전한' 판정: 경계를 MARGIN만큼 안쪽으로 수축한 뒤 within
    B_full = B.buffer(-MARGIN) if MARGIN > 0 else B
    hex_keep_5179 = hex_grid_5179[hex_grid_5179.within(B_full)].copy()

    # 센터 포인트는 보기용(클립 여부와 무관)
    centers_keep_5179 = hex_keep_5179.copy()
    centers_keep_5179["geometry"] = centers_keep_5179.geometry.centroid
else:
    # 센트로이드 기준: 경계를 CENTER_MARGIN만큼 수축 후, centroid가 내부면 keep
    B_in = B.buffer(-CENTER_MARGIN) if CENTER_MARGIN > 0 else B
    centers_all_5179 = hex_grid_5179[["hex_id", "geometry"]].copy()
    centers_all_5179["geometry"] = centers_all_5179.geometry.centroid
    centers_keep_5179 = centers_all_5179[centers_all_5179.within(B_in)].copy()

    # 원래 육각형 중, keep된 hex_id만 선택
    hex_keep_5179 = hex_grid_5179[hex_grid_5179["hex_id"].isin(centers_keep_5179["hex_id"])].copy()

# ================== 6) 라인(윤곽) 만들기: 클립 여부 선택 ==================
if CLIP_TO_BOUND:
    # 경계로 클립(경계 밖 라인은 잘림 → 시각적으로 깔끔)
    B_gdf = gpd.GeoDataFrame(geometry=[B], crs=hex_keep_5179.crs)
    hex_clip_5179 = gpd.overlay(hex_keep_5179, B_gdf, how="intersection", keep_geom_type=True)

    # 라인(윤곽)
    hex_lines_5179 = gpd.GeoDataFrame(
        {"hex_id": hex_clip_5179["hex_id"] if "hex_id" in hex_clip_5179 else range(len(hex_clip_5179))},
        geometry=hex_clip_5179.boundary,
        crs=hex_keep_5179.crs
    )
else:
    # 클립 없이 원형 라인(빠르지만 경계 밖 라인도 보임)
    hex_lines_5179 = gpd.GeoDataFrame(
        {"hex_id": hex_keep_5179["hex_id"]},
        geometry=hex_keep_5179.boundary,
        crs=hex_keep_5179.crs
    )

# ================== 7) Folium 시각화 ==================
hex_lines_4326 = hex_lines_5179.to_crs(CRS_LAT)
centers_4326   = centers_keep_5179.to_crs(CRS_LAT)

b = gdf_5179.to_crs(CRS_LAT).total_bounds
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]
m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")

folium.GeoJson(
    gdf_5179.to_crs(CRS_LAT).to_json(),
    name="Boundary",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False}
).add_to(m)

folium.GeoJson(
    hex_lines_4326.to_json(),
    name=f"Hex R={R}m (lines)",
    style_function=lambda f: {"color":"#555","weight":1,"fill":False}
).add_to(m)

# 센터 점(센트로이드/온전 기준으로 선택된 육각형들의 중심)
for pt in centers_4326.geometry:
    folium.CircleMarker(
        (pt.y, pt.x),
        radius=3, color="#fff", weight=1.2,
        fill=True, fillColor="#ff8a00", fillOpacity=0.95
    ).add_to(m)

folium.LayerControl().add_to(m)

print(
    f"[옵션] USE_FULL_WITHIN={USE_FULL_WITHIN}, CLIP_TO_BOUND={CLIP_TO_BOUND}, "
    f"MARGIN={MARGIN}, CENTER_MARGIN={CENTER_MARGIN}"
)
print(f"육각형(keep) 개수: {len(hex_keep_5179):,} | 라인 개수: {len(hex_lines_5179):,} | 센터 개수: {len(centers_keep_5179):,}")

m


## 수요지-공급지시각화

In [ ]:
import pandas as pd

df_filter = pd.read_csv('./data/1차필터링.csv')
df_filter

# folium 시각화

gdf_convenience_stores = gpd.GeoDataFrame(
    df_filter,
    geometry=gpd.points_from_xy(df_filter["lon"], df_filter["lat"]),
    crs="EPSG:4326"
)

for _, row in gdf_convenience_stores.iterrows():
    lat = row["lat"]
    lon = row["lon"]
    name = row["store_name"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 상호명 표시
    ).add_to(m)

m

In [ ]:
import folium

# 1) 파일 경로 지정 (여기만 바꾸세요)
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)
dunsan_4326

dunsan_5179 = dunsan.to_crs(5179)   


b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd

gdf1 = gpd.read_file('./data/다마_grid/grid_다마_100M.shp', encoding='utf-8')
gdf2 = gpd.read_file('./data/다바_grid/grid_다바_100M.shp', encoding='utf-8')
gdf3 = gpd.read_file('./data/라바_grid/grid_라바_100M.shp', encoding='utf-8')

grid_df = pd.concat([gdf1, gdf2, gdf3], ignore_index=True)
grid_df = grid_df.to_crs(5179)

grids_clip = gpd.clip(grid_df, dunsan_5179)
grids_clip

In [ ]:
grids_4326  = grids_clip.to_crs(4326)
dunsan_4326 = dunsan_5179.to_crs(4326) 

# 4) 맵 생성(경계 먼저, 격자 나중에 = 격자가 위)
b = dunsan_4326.total_bounds
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]
m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")

# 경계(아래)
folium.GeoJson(
    dunsan_4326.__geo_interface__,
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)

# (선택) 격자용 상위 pane으로 z-index 올리기
top = folium.map.CustomPane('top', z_index=650); top.add_to(m)

# 격자(위)
folium.GeoJson(
    grids_4326.__geo_interface__,
    name="격자",
    pane="top",
    style_function=lambda f: {"color":"#555","weight":0.6,"fill":False}
).add_to(m)


m


In [ ]:
df1 = pd.read_csv('./data/100M통계/2023년_인구_다마_100M.txt')
key_df = pd.Series(df1.columns, name='key').to_frame().reset_index(drop=True)
df1.columns = ['key']
df1 = pd.concat([key_df, df1], ignore_index=True)
parts1 = df1['key'].str.strip().str.split(r'\^', n=3, expand=True) 
parts1.columns = ['year', 'adm_code', 'indicator', 'suffix']

df2 = pd.read_csv('./data/100M통계/2023년_인구_다바_100M.txt')
key_df = pd.Series(df2.columns, name='key').to_frame().reset_index(drop=True)
df2.columns = ['key']
df2 = pd.concat([key_df, df2], ignore_index=True)
parts2 = df2['key'].str.strip().str.split(r'\^', n=3, expand=True)
parts2.columns = ['year', 'adm_code', 'indicator', 'suffix']

df3 = pd.read_csv('./data/100M통계/2023년_인구_라바_100M.txt')
key_df = pd.Series(df3.columns, name='key').to_frame().reset_index(drop=True)
df3.columns = ['key']
df3 = pd.concat([key_df, df3], ignore_index=True)
parts3 = df3['key'].str.strip().str.split(r'\^', n=3, expand=True)
parts3.columns = ['year', 'adm_code', 'indicator', 'suffix']

df_pop1 = pd.merge(gdf1, parts1,left_on='GRID_100M_', right_on = 'adm_code')
df_pop2 = pd.merge(gdf2, parts2,left_on='GRID_100M_', right_on = 'adm_code')
df_pop3 = pd.merge(gdf3, parts3,left_on='GRID_100M_', right_on = 'adm_code')

df_total1 = df_pop1[df_pop1['indicator'] == 'to_in_001']
df_total2 = df_pop2[df_pop2['indicator'] == 'to_in_001']
df_total3 = df_pop3[df_pop3['indicator'] == 'to_in_001']

df_total = pd.concat([df_total1, df_total2, df_total3], ignore_index=True)

In [ ]:
df_stat = pd.merge(grids_clip, df_total,left_on='GRID_100M_', right_on = 'GRID_100M_')
df_stat = df_stat[['GRID_100M_', 'geometry_x', 'suffix']]
df_stat.columns = ['GRID_100M_', 'geometry', 'suffix']
df_stat['suffix'] = pd.to_numeric(df_stat['suffix'])

In [ ]:
# df_stat에 필요한 열만
need = df_stat[['GRID_100M_', 'suffix']].drop_duplicates()

# 색칠 대상 격자만 추출 + 값 붙이기
sel = (
    grids_4326
      [grids_4326['GRID_100M_'].isin(need['GRID_100M_'])]
      .merge(need, on='GRID_100M_', how='left')
      .to_crs(4326)
)


from branca.colormap import linear

# 회색 윤곽의 원본 격자는 이미 추가되어 있다고 했으니 그대로 두고,
# df_stat 해당 격자만 채우기
vmin, vmax = float(sel['suffix'].min()), float(sel['suffix'].max())
cmap = linear.viridis.scale(vmin, vmax)

def style_by_suffix(feature):
    s = feature['properties'].get('suffix', None)
    fill = cmap(s) if s is not None else '#cccccc'
    return {"color":"#666","weight":0.3,"fill":True,"fillColor":fill,"fillOpacity":0.65}

folium.GeoJson(
    sel.__geo_interface__,
    name="선택 격자(색=suffix)",
    pane="top",
    style_function=style_by_suffix,
    tooltip=folium.GeoJsonTooltip(
        fields=["GRID_100M_", "suffix"],
        aliases=["그리드", "suffix"],
        localize=True
    )
).add_to(m)

cmap.caption = "suffix"
cmap.add_to(m)
m

In [ ]:
hex_lines_4326 = hex_lines_5179.to_crs(CRS_LAT)
centers_4326   = centers_keep_5179.to_crs(CRS_LAT)

b = gdf_5179.to_crs(CRS_LAT).total_bounds
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

folium.GeoJson(
    gdf_5179.to_crs(CRS_LAT).to_json(),
    name="Boundary",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False}
).add_to(m)

folium.GeoJson(
    hex_lines_4326.to_json(),
    name=f"Hex R={R}m (lines)",
    style_function=lambda f: {"color":"#555","weight":1,"fill":False}
).add_to(m)

for pt in centers_4326.geometry:
    folium.CircleMarker(
        (pt.y, pt.x),
        radius=3, color="#fff", weight=1.2,
        fill=True, fillColor="#ff8a00", fillOpacity=0.95
    ).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:
gdf_convenience_stores = gpd.GeoDataFrame(
    df_filter,
    geometry=gpd.points_from_xy(df_filter["lon"], df_filter["lat"]),
    crs="EPSG:4326"
)

for _, row in gdf_convenience_stores.iterrows():
    lat = row["lat"]
    lon = row["lon"]
    name = row["store_name"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.7,
        tooltip=name  # 마우스 올리면 상호명 표시
    ).add_to(m)

m

## 수요지 격자 교집합

In [ ]:
pd.set_option('display.max_rows', 10)

hex_m = (hex_clip_5179 if CLIP_TO_BOUND else hex_keep_5179).copy()
hex_m = hex_m[['hex_id','geometry']].to_crs(CRS_M)

# 2) sanity check: 폴리곤만 남기기
hex_m = hex_m[hex_m.geometry.geom_type.isin(['Polygon','MultiPolygon'])].copy()

# 3) (권장) 너무 작은 조각 제거
hex_m['area'] = hex_m.geometry.area
hex_m = hex_m[hex_m['area'] > 50].drop(columns='area')  # 임계값 필요시 조정
grids_m = df_stat.to_crs(5179).copy()
grid_area = grids_m.set_index('GRID_100M_').geometry.area

In [ ]:
# 교집합 계산

import geopandas as gpd

# 1) 교집합(육각형 ∩ 격자)
inter = gpd.overlay(
    hex_m[['hex_id','geometry']],
    grids_m[['GRID_100M_','suffix','geometry']],
    how='intersection'
)

# 2) 교집합 면적과 가중치(= 교집합면적 / 격자면적)
inter['inter_area'] = inter.geometry.area
inter['grid_area']  = inter['GRID_100M_'].map(grid_area)
inter['w']          = inter['inter_area'] / inter['grid_area']

# 3) 가중 기여값
inter['suffix_contrib'] = inter['suffix'] * inter['w']
inter.head()

In [ ]:
# 면적비 가중 합 (추천)
pd.set_option('display.max_rows', 30)

hex_suffix_aw = (
    inter.groupby('hex_id', as_index=False)['suffix_contrib']
         .sum().rename(columns={'suffix_contrib':'suffix_aw_sum'})
)

hex_suffix_aw

# hex_m: 육각형 폴리곤(5179), inter: 교집합 결과, hex_suffix_aw: 방금 만든 테이블
hex_dem = hex_m[['hex_id','geometry']].merge(hex_suffix_aw, on='hex_id', how='left')
hex_dem['suffix_aw_sum'] = hex_dem['suffix_aw_sum'].fillna(0)

# 중심점(수요지 포인트)
demand_m = hex_dem.copy()
demand_m['geometry'] = demand_m.geometry.centroid  # 5179(미터) 좌표계 유지
demand_m = demand_m[['hex_id','suffix_aw_sum','geometry']]

# Folium 용 위경도도 필요하면
demand_ll = demand_m.to_crs(4326).copy()
demand_ll['lon'] = demand_ll.geometry.x
demand_ll['lat'] = demand_ll.geometry.y

demand_ll

In [ ]:
demand_ll.to_csv('./data/수요지데이터.csv', index=False)
